### **Chapter 9.5: Robustness, Regularization, and the LTI Boundary**

Willems' fundamental lemma is exact for controllable LTI systems. The bumpy Mountain-Car terrain (`case = 3`) is nonlinear because the slope depends on position. Relative to the flat model it can be interpreted as an **unmodeled state-dependent perturbation**.

This notebook uses the bumpy case for two tutorial demonstrations:

1. as the bump amplitude grows, a Hankel matrix collected on the flat LTI system no longer reconstructs the true nonlinear trajectories exactly;
2. hard deterministic DeePC can become brittle under model mismatch, while soft output-history fitting and $g$-regularization provide a simple robustification that still yields an OSQP QP.

We do **not** claim that the LTI fundamental lemma remains exact for the nonlinear terrain.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.env import *
from ex5_MPC.mpc_utils import LinearMPCController
from ex9_DeePC.deepc_utils import *

### **Part 1: Flat Offline Data**

Both DeePC variants below use a dataset collected on the nominal flat system. The bumpy plant is introduced only during testing/deployment.

In [ ]:
freq = 20
dt = 1.0 / freq
N = 20
T_ini = 4

initial_state = np.array([-0.5, 0.0])
target_state = np.array([0.6, 0.0])

env_flat = Env(1, initial_state, target_state, input_lbs=-1.0, input_ubs=1.0)
dynamics_flat = Dynamics(env_flat)

u_data, y_data = collect_deepc_data(
    env_flat,
    dynamics_flat,
    freq=freq,
    n_samples=400,
    excitation_amplitude=0.8,
    initial_state=env_flat.target_state,
    seed=1,
)

Q = np.diag([1.0, 1.0])
R = np.array([[0.1]])
Qf = Q

### **Part 2: Where Exact Hankel Reconstruction Breaks Down**

Use the same test input and initial condition for a family of terrains

$$
h(p)=k\cos(18p).
$$

At $k=0$ the system reduces to the flat LTI case. Increasing $k$ increases the nonlinear perturbation. We attempt to reconstruct every resulting trajectory using **only the flat-system Hankel matrix**.

In [ ]:
def rollout(dynamics, x0, u_sequence, dt):
    x = np.asarray(x0, dtype=float).copy()
    states = [x.copy()]
    for u in np.asarray(u_sequence):
        x = dynamics.one_step_forward(x, np.asarray(u).reshape(-1), dt)
        states.append(x.copy())
    return np.asarray(states)

L = 16
rng = np.random.default_rng(30)
u_test = rng.uniform(-0.45, 0.45, size=(L, 1))
x0_test = np.array([-0.35, 0.18])

bump_amplitudes = np.array([0.0, 0.001, 0.0025, 0.005, 0.0075, 0.01])
reconstruction_errors = []

for bump in bump_amplitudes:
    env_bumpy = Env(3, initial_state, target_state, param=float(bump), input_lbs=-1.0, input_ubs=1.0)
    dynamics_bumpy = Dynamics(env_bumpy)
    y_test = rollout(dynamics_bumpy, x0_test, u_test, dt)

    _, _, _, error = reconstruct_trajectory(
        u_data,
        y_data,
        u_test,
        y_test,
        length=L,
        input_offset=np.zeros(1),
        output_offset=env_flat.target_state,
    )
    reconstruction_errors.append(error)

plt.figure(figsize=(7, 4))
plt.semilogy(bump_amplitudes, np.maximum(reconstruction_errors, 1e-16), marker="o")
plt.xlabel("Bump amplitude k")
plt.ylabel("Relative Hankel reconstruction error")
plt.title("Breakdown of exact LTI behavioral representation")
plt.tight_layout()
plt.show()

### **Part 3: Minimal Regularized DeePC**

The deterministic controller imposes exact output-history consistency:

$$
Y_p g = y_{\mathrm{ini}}.
$$

Under disturbances, noise, or nonlinear mismatch, that equality can be too brittle. The regularized controller keeps the past input equality hard but moves output consistency into the objective:

$$
\min_g\; J_{\rm track}(g)
+\lambda_g\|g\|_2^2
+\lambda_y\|Y_{\rm init}g-y_{\rm meas}\|_2^2.
$$

This remains a convex quadratic program in $g$, so the same OSQP backend is used.

In [ ]:
def make_vanilla_deepc():
    return DeePCController(
        env_flat, dynamics_flat, u_data, y_data,
        Q, R, Qf, freq, N,
        T_ini=T_ini,
        lambda_g=1e-8,
        lambda_y=None,
        history_initialization='equilibrium',
        name='Vanilla_DeePC',
        verbose=False,
    )

def make_regularized_deepc(lambda_g=1e-3, lambda_y=1e3):
    return DeePCController(
        env_flat, dynamics_flat, u_data, y_data,
        Q, R, Qf, freq, N,
        T_ini=T_ini,
        lambda_g=lambda_g,
        lambda_y=lambda_y,
        history_initialization='equilibrium',
        name='Regularized_DeePC',
        verbose=False,
    )

def make_nominal_mpc():
    return LinearMPCController(
        env_flat, dynamics_flat, Q, R, Qf, freq, N,
        name='Nominal_Linear_MPC',
        verbose=False,
    )

### **Part 4: Deploy the Nominal Controllers on a Bumpy Plant**

All three controllers are designed from the flat system/data. Only the plant used for propagation is changed.

If a hard DeePC solve becomes infeasible, the runner records the failure, applies the equilibrium fallback input for that step, and reinitializes the history before trying again. The failure count is itself one of the quantities of interest.

In [ ]:
def run_on_plant(controller, plant_dynamics, env_plant, t_terminal=6.0, measurement_noise_std=0.0, seed=0):
    rng = np.random.default_rng(seed)
    x = env_plant.init_state.copy()
    states = [x.copy()]
    inputs = []
    failures = 0

    for k in range(int(freq * t_terminal)):
        measured_state = x + rng.normal(0.0, measurement_noise_std, size=x.shape)
        try:
            u = controller.compute_action(measured_state, k)
            if isinstance(u, tuple):
                u = u[0]
        except RuntimeError:
            failures += 1
            u = np.atleast_1d(dynamics_flat.get_equilibrium_input(env_flat.target_state))
            if isinstance(controller, DeePCController):
                controller.initialize_history(measured_state, mode='equilibrium')

        u = np.asarray(u, dtype=float).reshape(-1)
        x = plant_dynamics.one_step_forward(x, u, 1.0/freq)
        states.append(x.copy())
        inputs.append(u.copy())

    return np.asarray(states), np.asarray(inputs), failures

def closed_loop_cost(states, inputs):
    dx = states - env_flat.target_state
    u_ref = np.atleast_1d(dynamics_flat.get_equilibrium_input(env_flat.target_state))
    du = inputs - u_ref
    J = sum(dx[k] @ Q @ dx[k] + du[k] @ R @ du[k] for k in range(len(inputs)))
    J += dx[-1] @ Qf @ dx[-1]
    return float(J)

showcase_bump = 0.005
env_bumpy = Env(3, initial_state, target_state, param=showcase_bump, input_lbs=-1.0, input_ubs=1.0)
dynamics_bumpy = Dynamics(env_bumpy)

x_mpc, u_mpc, fail_mpc = run_on_plant(make_nominal_mpc(), dynamics_bumpy, env_bumpy)
x_van, u_van, fail_van = run_on_plant(make_vanilla_deepc(), dynamics_bumpy, env_bumpy)
x_reg, u_reg, fail_reg = run_on_plant(make_regularized_deepc(), dynamics_bumpy, env_bumpy)

print("QP failures / fallbacks")
print("  nominal MPC:       ", fail_mpc)
print("  vanilla DeePC:     ", fail_van)
print("  regularized DeePC: ", fail_reg)
print("closed-loop costs")
print("  nominal MPC:       ", closed_loop_cost(x_mpc, u_mpc))
print("  vanilla DeePC:     ", closed_loop_cost(x_van, u_van))
print("  regularized DeePC: ", closed_loop_cost(x_reg, u_reg))

In [ ]:
t_x = np.arange(len(x_mpc)) / freq
t_u = np.arange(len(u_mpc)) / freq

fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for x, label in [(x_mpc, "nominal MPC"), (x_van, "vanilla DeePC"), (x_reg, "regularized DeePC")]:
    ax[0].plot(t_x, x[:, 0], label=label)
    ax[1].plot(t_x, x[:, 1], label=label)
ax[0].axhline(env_flat.target_position, linestyle=":", label="target")
ax[0].set_ylabel("position")
ax[0].legend()
ax[1].set_ylabel("velocity")

ax[2].plot(t_u, u_mpc[:, 0], label="nominal MPC")
ax[2].plot(t_u, u_van[:, 0], label="vanilla DeePC")
ax[2].plot(t_u, u_reg[:, 0], label="regularized DeePC")
ax[2].set_ylabel("input")
ax[2].set_xlabel("Time (s)")

fig.suptitle(f"Controllers designed on flat data, deployed at bump amplitude k={showcase_bump}")
plt.tight_layout()
plt.show()

### **Part 5: Sweep the Perturbation Strength**

The purpose of this sweep is not to assume that one controller must always win. Instead, it exposes three distinct effects as the LTI assumption is violated:

- tracking degradation;
- hard-DeePC infeasibility/fallbacks;
- the effect of regularization on robustness.

In [ ]:
sweep_bumps = np.array([0.0, 0.0025, 0.005, 0.0075, 0.01])
metrics = {
    "MPC": {"cost": [], "terminal": [], "failures": []},
    "Vanilla DeePC": {"cost": [], "terminal": [], "failures": []},
    "Regularized DeePC": {"cost": [], "terminal": [], "failures": []},
}

for bump in sweep_bumps:
    env_plant = Env(3, initial_state, target_state, param=float(bump), input_lbs=-1.0, input_ubs=1.0)
    plant = Dynamics(env_plant)

    runs = {
        "MPC": run_on_plant(make_nominal_mpc(), plant, env_plant, t_terminal=5.0),
        "Vanilla DeePC": run_on_plant(make_vanilla_deepc(), plant, env_plant, t_terminal=5.0),
        "Regularized DeePC": run_on_plant(make_regularized_deepc(), plant, env_plant, t_terminal=5.0),
    }

    for name, (x, u, failures) in runs.items():
        metrics[name]["cost"].append(closed_loop_cost(x, u))
        metrics[name]["terminal"].append(abs(x[-1, 0] - env_flat.target_position))
        metrics[name]["failures"].append(failures)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for name, result in metrics.items():
    ax[0].plot(sweep_bumps, result["cost"], marker="o", label=name)
    ax[1].plot(sweep_bumps, result["terminal"], marker="o", label=name)
    ax[2].plot(sweep_bumps, result["failures"], marker="o", label=name)

ax[0].set_xlabel("Bump amplitude k")
ax[0].set_ylabel("closed-loop cost")
ax[0].set_title("Tracking performance")

ax[1].set_xlabel("Bump amplitude k")
ax[1].set_ylabel("terminal position error")
ax[1].set_title("Terminal accuracy")

ax[2].set_xlabel("Bump amplitude k")
ax[2].set_ylabel("QP failures / fallbacks")
ax[2].set_title("Brittleness of hard matching")
ax[2].legend()

plt.tight_layout()
plt.show()

### **Optional: Measurement-Noise Stress Test**

The same runner can inject Gaussian measurement noise before calling the controller. This is useful if the tutorial should separate **plant nonlinearity** from **measurement inconsistency**:

```python
x_reg_noise, u_reg_noise, failures = run_on_plant(
    make_regularized_deepc(lambda_g=1e-3, lambda_y=1e3),
    dynamics_flat,
    env_flat,
    measurement_noise_std=0.01,
)
```

For noisy data, `lambda_y` controls how strongly DeePC trusts the measured output history, while `lambda_g` regularizes the behavioral coefficient.

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway: Exact behavioral representation has a boundary**

The LTI fundamental lemma gives an exact finite-data representation only under its assumptions. Bumpy terrain provides a controlled way to move outside those assumptions. Regularization does not make the nonlinear system LTI; it makes the data-driven optimization less brittle when measured trajectories are not exactly consistent with the nominal Hankel behavior.
</blockquote>

**References:** Coulson, Lygeros, and Dörfler, *Data-Enabled Predictive Control: In the Shallows of the DeePC*; subsequent robust DeePC work on regularization and noisy data.